# TP2 - Referencia para merge de destinos canonicos

Esta notebook deja una forma reproducible de aplicar `destination_with_nearest.csv` sobre los historicos del TP.

Objetivos:

- usar una unica funcion de pretratamiento, compartida con `scripts/destination_mapping_preprocess.py`;
- construir `destination_final` y `destination_name` sin modificar manualmente el dataset;
- medir cobertura por filas, por demanda ponderada (`count_repeated`) y por geografias unicas;
- distinguir matches exactos de fallbacks auditables mediante `match_level`;
- listar los principales no-matcheados por demanda.

La salida esperada no es necesariamente 100% con los CSV actuales. El objetivo es mapear todo lo que se puede mapear de forma defendible, dejar trazabilidad de la regla usada y auditar lo que queda fuera.

El mismo procedimiento puede ejecutarse desde consola:

```bash
python3 scripts/destination_mapping_preprocess.py --data-dir data --mapping data/destination_with_nearest.csv
```


In [1]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.3f}'.format)

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / 'scripts'))

from destination_mapping_preprocess import (  # noqa: E402
    GEO_COLS,
    US_STATE_CODES,
    aggregate_historical_geographies,
    apply_destination_mapping_preprocessed,
    clean_text,
    coverage_tables,
    top_unmatched,
)

DATA_DIR = ROOT / 'data'
MAPPING_PATH = DATA_DIR / 'destination_with_nearest.csv'

# Para validar rapido el merge fila a fila. La cobertura final se calcula mas abajo por chunks.
NROWS_PER_FILE = 250_000
FULL_COVERAGE_CHUNKSIZE = 500_000


## 1. Carga de datos

Para revisar rapidamente ejemplos fila a fila usamos una muestra de cada archivo. Para el coverage final no usamos muestra: agregamos todo el historico por geografia y procesamos esa tabla agregada.

In [2]:
historical_files = sorted(DATA_DIR.glob('datos_historicos_*.csv'))
if not historical_files:
    raise FileNotFoundError(f'No se encontraron datos_historicos_*.csv en {DATA_DIR}')

mapping_df = pd.read_csv(
    MAPPING_PATH,
    dtype={'nearest_destination_id': str, 'reference': str, 'city': str},
)

sample_parts = []
usecols = GEO_COLS + ['count_repeated']
for file in historical_files:
    sample_parts.append(
        pd.read_csv(
            file,
            usecols=usecols,
            nrows=NROWS_PER_FILE,
            dtype={col: str for col in GEO_COLS},
            low_memory=False,
        )
    )

df_raw = pd.concat(sample_parts, ignore_index=True)

print('Archivos historicos:')
for file in historical_files:
    print(f'- {file.name}')
print(f'Filas de muestra cargadas: {len(df_raw):,}')
print(f'Referencias en mapping: {len(mapping_df):,}')

display(df_raw.head())
display(mapping_df.head())


Archivos historicos:
- datos_historicos_2024.csv
- datos_historicos_2025.csv
Filas de muestra cargadas: 500,000
Referencias en mapping: 15,988


,city,state,country,country_code,count_repeated
0,남구,Busan,South Korea,kr,2
1,동구,Busan,South Korea,kr,1
2,동구,Busan,South Korea,kr,2
3,동구,Busan,South Korea,kr,1
4,동구,Busan,South Korea,kr,1


,reference,city,latitude,longitude,nearest_destination_id,nearest_destination_name
0,AD - Andorra La Bella,Andorra La Bella,42.507,1.531,50643,Andorra la Vella
1,AD - Andorra La Vella,Andorra La Vella,42.509,1.529,50643,Andorra la Vella
2,AD - Arinsal,Arinsal,42.576,1.481,50643,Andorra la Vella
3,AD - El Pas de la Casa,El Pas de la Casa,42.542,1.734,21795,Andorra
4,AD - Escaldes-Engordany,Escaldes-Engordany,42.510,1.537,50643,Andorra la Vella


## 2. Pretratamiento correcto del mapping

El error que puede llevar a 0% de match no esta en usar `destination_with_nearest.csv`, sino en construir una sola clave rigida. En los historicos `country_code` viene en minuscula (`us`, `kr`) y el mapping usa mayuscula (`US`, `KR`). Ademas, el mapping tiene referencias de dos y tres partes:

- `US - CA - Los Angeles`, con estado;
- `KR - Busan`, sin estado;
- `AR - Buenos Aires`, sin estado.

El pretratamiento recomendado hace esto:

1. normaliza texto, espacios, acentos y mayusculas;
2. convierte estados de Estados Unidos de nombre completo a codigo (`Florida` -> `FL`);
3. prueba claves exactas `country-state_code-city`, `country-state_name-city` y `country-city` siempre contra `(reference, city)`;
4. agrega normalizaciones puntuales auditables, como alias `USA`, territorios de Estados Unidos y sufijo `CDP`;
5. permite `country-state` como ciudad solo para paises no-US, porque en estos historicos a veces `city` es barrio/distrito y `state` contiene el mercado grande;
6. deja cada asignacion marcada en `match_level`.

No usamos como asignacion automatica general los fallbacks `country-state` para Estados Unidos ni `(country, city)` ignorando el estado, porque pueden crear falsos matches.

In [3]:
reference_parts = (
    mapping_df['reference']
    .astype(str)
    .str.count(' - ')
    .add(1)
    .value_counts()
    .sort_index()
    .rename_axis('reference_parts')
    .reset_index(name='n_references')
)

display(reference_parts)

display(
    mapping_df[['reference', 'city', 'nearest_destination_id', 'nearest_destination_name']]
    .sample(10, random_state=42)
)


,reference_parts,n_references
0,1,9
1,2,8403
2,3,7561
3,4,15


,reference,city,nearest_destination_id,nearest_destination_name
15977,ZA - Waterval Boven,Waterval Boven,50536,Nelspruit
11242,US - IL - Wheeling,Wheeling,673,Chicago
150,AT - Ehrwald,Ehrwald,22133,Garmisch-Partenkirchen
1684,CA - ON - Blue Mountains,Blue Mountains,50915,Collingwood
7496,NL - Scheveningen,Scheveningen,5436,The Hague
5205,IE - Ballinalee,Ballinalee,5042,Athlone
1034,BR - Confins,Confins,26539,Belo Horizonte
123,AR - Tunuyan,Tunuyan,52093,Lujan de Cuyo
88,AR - Mina Clavero,Mina Clavero,22281,Córdoba
5380,IM - Port Erin,Port Erin,5017,Isle of Man


## 3. Aplicacion sobre muestra

Esta seccion sirve para inspeccionar ejemplos concretos. No debe usarse para reportar conclusiones finales de cobertura.

In [4]:
df_mapped = apply_destination_mapping_preprocessed(df_raw, mapping_df)

sample_geo = (
    df_raw
    .groupby(GEO_COLS, dropna=False)
    .agg(row_count=('city', 'size'), demand_weight=('count_repeated', 'sum'))
    .reset_index()
)
sample_geo_mapped = apply_destination_mapping_preprocessed(sample_geo, mapping_df)

sample_summary, sample_by_level = coverage_tables(sample_geo_mapped)

display(sample_summary)
display(sample_by_level)

display(
    df_mapped[
        ['country_code', 'country', 'state', 'city', 'destination_final', 'destination_name', 'match_level']
    ].head(20)
)


,metric,mapped,total,coverage_pct
0,rows,365693,500000,73.139
1,demand_weight,2948018,3442410,85.638
2,unique_geographies,1457,4192,34.757


,rows,demand_weight,unique_geographies,row_pct,demand_pct
match_level,,,,,
cc_country_city,168607,1356483,552,33.721,39.405
cc_state_code_city,155180,1306317,628,31.036,37.948
no_match,134307,494392,2735,26.861,14.362
cc_state_name_city,20113,220201,6,4.023,6.397
non_us_state_as_place,21793,65017,271,4.359,1.889


,country_code,country,state,city,destination_final,destination_name,match_level
0,kr,South Korea,Busan,남구,4615,Busan,non_us_state_as_place
1,kr,South Korea,Busan,동구,4615,Busan,non_us_state_as_place
2,kr,South Korea,Busan,동구,4615,Busan,non_us_state_as_place
3,kr,South Korea,Busan,동구,4615,Busan,non_us_state_as_place
4,kr,South Korea,Busan,동구,4615,Busan,non_us_state_as_place
5,kr,South Korea,Busan,동구,4615,Busan,non_us_state_as_place
6,kr,South Korea,Busan,동구,4615,Busan,non_us_state_as_place
7,kr,South Korea,Busan,부암동,4615,Busan,non_us_state_as_place
8,kr,South Korea,Busan,부암동,4615,Busan,non_us_state_as_place
9,kr,South Korea,Busan,수영구,4615,Busan,non_us_state_as_place


## 4. Coverage exacto sobre todo el historico

Para reportar coverage final no hace falta cargar los 5 millones de filas completas en memoria. Primero agregamos por geografia cruda en chunks y despues aplicamos el mapping sobre esa tabla agregada.

Leer las metricas asi:

- `rows`: filas del CSV historico que recibieron `nearest_destination_id`. Una fila es una combinacion agregada del dataset, no necesariamente una busqueda individual.
- `demand_weight`: suma de `count_repeated` de las filas mapeadas. Esta es la metrica mas importante para entender cobertura de demanda.
- `unique_geographies`: combinaciones crudas unicas de `country_code`, `country`, `state` y `city`. No mide negocio ni busquedas; mide la cola de nombres/lugares que aparece en el input. Por eso puede ser mucho mas bajo aunque la demanda mapeada sea alta.

Si `unique_geographies` da cerca de 35%, no significa que solo cubrimos 35% del negocio. Significa que hay muchas geografias crudas poco frecuentes o con nombres administrativos que quedan para auditar.


In [5]:
geo_counts = aggregate_historical_geographies(historical_files, chunksize=FULL_COVERAGE_CHUNKSIZE)
geo_counts_mapped = apply_destination_mapping_preprocessed(geo_counts, mapping_df)

full_summary, full_by_level = coverage_tables(geo_counts_mapped)

display(full_summary)
display(full_by_level)

display(top_unmatched(geo_counts_mapped, n=30))


,metric,mapped,total,coverage_pct
0,rows,3558663,5158190,68.991
1,demand_weight,29916231,37167072,80.491
2,unique_geographies,14767,41535,35.553


,rows,demand_weight,unique_geographies,row_pct,demand_pct
match_level,,,,,
cc_state_code_city,1708954,17348126,6983,33.131,46.676
cc_country_city,1479587,10035493,5220,28.684,27.001
no_match,1599527,7250841,26768,31.009,19.509
cc_state_name_city,113200,1294177,26,2.195,3.482
non_us_state_as_place,228126,918793,2517,4.423,2.472
cc_usa_alias_state_code_city,16854,192935,3,0.327,0.519
cc_cdp_state_code_city,11942,126707,18,0.232,0.341


,country_code,country,state,city,row_count,demand_weight
35059,us,United States of America,Florida,Miami Beach,19706,473159
37536,us,United States of America,Nevada,Winchester,16900,386531
41534,NaN,NaN,NaN,NaN,26649,373051
34078,us,United States of America,California,Lomita Park,11188,279213
40600,us,United States of America,Washington,SeaTac,8200,188259
34885,us,United States of America,Florida,Dr. Phillips,14201,180182
39258,us,United States of America,Puerto Rico,San Juan Antiguo,11875,168976
35895,us,United States of America,Illinois,Norridge,8453,159172
19990,nl,Netherlands,NaN,NaN,13494,113172
35639,us,United States of America,Hawaii,Kahaluu-Keauhou CDP,9335,96614


## 5. Comparacion de estrategias

Esta tabla separa tres cosas:

- el problema original de formato;
- lo que se recupera con una cascada estricta de claves del mapping;
- lo que agrega el pretratamiento recomendado.

Los fallbacks marcados como `no_recomendado` sirven para diagnostico y auditoria, no para asignacion automatica final.

In [6]:
def summarize_mask(name, mask, source_geo):
    total_rows = source_geo['row_count'].sum()
    total_demand = source_geo['demand_weight'].sum()
    return {
        'strategy': name,
        'mapped_rows': source_geo.loc[mask, 'row_count'].sum(),
        'row_coverage_pct': source_geo.loc[mask, 'row_count'].sum() / total_rows * 100,
        'mapped_demand': source_geo.loc[mask, 'demand_weight'].sum(),
        'demand_coverage_pct': source_geo.loc[mask, 'demand_weight'].sum() / total_demand * 100,
        'mapped_geographies': int(mask.sum()),
        'geo_coverage_pct': mask.mean() * 100,
    }

raw_mapping_pairs = set(
    mapping_df[['reference', 'city']]
    .fillna('')
    .astype(str)
    .itertuples(index=False, name=None)
)

comparison_geo = geo_counts.copy()
comparison_geo['state_clean'] = comparison_geo['state'].map(clean_text)
comparison_geo['state_code'] = comparison_geo['state_clean'].map(US_STATE_CODES)
comparison_geo['original_reference'] = (
    comparison_geo['country_code'].fillna('')
    + ' - '
    + comparison_geo['state_code'].fillna('')
    + ' - '
    + comparison_geo['city'].fillna('')
)
comparison_geo['upper_reference'] = (
    comparison_geo['country_code'].fillna('').str.upper()
    + ' - '
    + comparison_geo['state_code'].fillna('')
    + ' - '
    + comparison_geo['city'].fillna('')
)

original_match = comparison_geo.apply(
    lambda row: (str(row['original_reference']), str(row['city'])) in raw_mapping_pairs,
    axis=1,
)
upper_state_city_match = comparison_geo.apply(
    lambda row: (str(row['upper_reference']), str(row['city'])) in raw_mapping_pairs,
    axis=1,
)

strict_mapped = apply_destination_mapping_preprocessed(
    geo_counts,
    mapping_df,
    use_usa_alias=False,
    recode_us_territories=False,
    use_cdp_city_variant=False,
    use_non_us_state_as_place=False,
)
recommended_mapped = geo_counts_mapped

mapping_for_tuple = mapping_df.copy()
mapping_for_tuple['ref_clean'] = mapping_for_tuple['reference'].map(clean_text)
mapping_for_tuple['city_clean'] = mapping_for_tuple['city'].map(clean_text)
mapping_for_tuple['country_clean'] = mapping_for_tuple['ref_clean'].str.split(' - ').str[0]
country_city_pairs = set(zip(mapping_for_tuple['country_clean'], mapping_for_tuple['city_clean']))
comparison_geo['country_clean'] = comparison_geo['country_code'].fillna('').str.upper().map(clean_text)
comparison_geo['city_clean'] = comparison_geo['city'].map(clean_text)
country_city_tuple_match = pd.Series(
    list(zip(comparison_geo['country_clean'], comparison_geo['city_clean'])),
    index=comparison_geo.index,
).isin(country_city_pairs)

strategy_comparison = pd.DataFrame([
    summarize_mask('funcion_original_repo', original_match, comparison_geo),
    summarize_mask('solo_upper_y_state_city', upper_state_city_match, comparison_geo),
    summarize_mask('cascada_estricta_reference_city', strict_mapped['is_mapped'], comparison_geo),
    summarize_mask('pretratamiento_recomendado', recommended_mapped['is_mapped'], comparison_geo),
    summarize_mask(
        'fallback_pais_ciudad_ignorando_estado_no_recomendado',
        recommended_mapped['is_mapped'] | country_city_tuple_match,
        comparison_geo,
    ),
])

display(strategy_comparison)


,strategy,mapped_rows,row_coverage_pct,mapped_demand,demand_coverage_pct,mapped_geographies,geo_coverage_pct
0,funcion_original_repo,0,0.000,0,0.000,0,0.000
1,solo_upper_y_state_city,1703591,33.027,17302596,46.554,6969,16.779
2,cascada_estricta_reference_city,3266882,63.334,28537126,76.781,12158,29.272
3,pretratamiento_recomendado,3558663,68.991,29916231,80.491,14767,35.553
4,fallback_pais_ciudad_ignorando_estado_no_recom...,3631235,70.397,30498873,82.059,15781,37.994


## 6. Ejemplos de fallbacks peligrosos

Estos ejemplos muestran por que no conviene maximizar coverage sin mirar calidad del match. Si un fallback aumenta coverage pero mueve una localidad a un mercado incorrecto, contamina el baseline.

In [7]:
lookup = (
    mapping_df.assign(
        ref_clean=mapping_df['reference'].map(clean_text),
        city_clean=mapping_df['city'].map(clean_text),
    )
    .drop_duplicates(['ref_clean', 'city_clean'])
    .set_index(['ref_clean', 'city_clean'])[['nearest_destination_id', 'nearest_destination_name', 'reference']]
)

unsafe_geo = comparison_geo.copy()
unsafe_geo['ref_country_state_clean'] = (
    unsafe_geo['country_code'].fillna('').str.upper() + ' - ' + unsafe_geo['state'].fillna('')
).map(clean_text)
unsafe_state_idx = pd.MultiIndex.from_arrays([
    unsafe_geo['ref_country_state_clean'],
    unsafe_geo['state_clean'],
])
unsafe_geo['unsafe_state_destination_id'] = unsafe_state_idx.map(lookup['nearest_destination_id'])
unsafe_geo['unsafe_state_destination_name'] = unsafe_state_idx.map(lookup['nearest_destination_name'])
unsafe_geo['unsafe_state_reference_used'] = unsafe_state_idx.map(lookup['reference'])

unsafe_state_candidates = (
    unsafe_geo[
        unsafe_geo['unsafe_state_destination_id'].notna()
        & ~recommended_mapped['is_mapped']
    ]
    .sort_values('demand_weight', ascending=False)
)

display(
    unsafe_state_candidates[
        [
            'country_code', 'country', 'state', 'city', 'unsafe_state_reference_used',
            'unsafe_state_destination_id', 'unsafe_state_destination_name', 'row_count', 'demand_weight'
        ]
    ].head(20)
)

tuple_lookup = (
    mapping_for_tuple
    .drop_duplicates(['country_clean', 'city_clean'])
    .set_index(['country_clean', 'city_clean'])[['nearest_destination_id', 'nearest_destination_name', 'reference']]
)
tuple_idx = pd.MultiIndex.from_arrays([comparison_geo['country_clean'], comparison_geo['city_clean']])
comparison_geo['tuple_destination_id'] = tuple_idx.map(tuple_lookup['nearest_destination_id'])
comparison_geo['tuple_destination_name'] = tuple_idx.map(tuple_lookup['nearest_destination_name'])
comparison_geo['tuple_reference_used'] = tuple_idx.map(tuple_lookup['reference'])

unsafe_country_city = (
    comparison_geo[
        country_city_tuple_match
        & ~recommended_mapped['is_mapped']
    ]
    .sort_values('demand_weight', ascending=False)
)

display(
    unsafe_country_city[
        [
            'country_code', 'country', 'state', 'city', 'tuple_reference_used',
            'tuple_destination_id', 'tuple_destination_name', 'row_count', 'demand_weight'
        ]
    ].head(20)
)


,country_code,country,state,city,unsafe_state_reference_used,unsafe_state_destination_id,unsafe_state_destination_name,row_count,demand_weight
34078,us,United States of America,California,Lomita Park,US - California,645,Los Angeles,11188,279213
40600,us,United States of America,Washington,SeaTac,US - Washington,657,Washington DC,8200,188259
40483,us,United States of America,Washington,Boulevard Park,US - Washington,657,Washington DC,4158,41372
40488,us,United States of America,Washington,Burien,US - Washington,657,Washington DC,2464,20259
33191,us,United States,Washington,SeaTac,US - Washington,657,Washington DC,2005,9610
34209,us,United States of America,California,Paramount,US - California,645,Los Angeles,1338,5359
34366,us,United States of America,California,South San Francisco,US - California,645,Los Angeles,1536,5331
40487,us,United States of America,Washington,Bryn Mawr-Skyway,US - Washington,657,Washington DC,1234,3923
26309,us,USA,Missouri,St. Ann,US - Missouri,22925,Kansas City,873,3890
37330,us,United States of America,Missouri,Hampton,US - Missouri,22925,Kansas City,992,3678


,country_code,country,state,city,tuple_reference_used,tuple_destination_id,tuple_destination_name,row_count,demand_weight
37536,us,United States of America,Nevada,Winchester,US - KY - Winchester,25316,Lexington,16900,386531
40377,us,United States of America,Virginia,Oak Grove,US - KY - Oak Grove,799,Nashville,3766,35333
35095,us,United States of America,Florida,Oak Ridge,US - TN - Oak Ridge,32982,Knoxville,4999,22778
35298,us,United States of America,Florida,Williamsburg,US - IA - Williamsburg,24146,Madison,2320,11967
34898,us,United States of America,Florida,Edgewood,US - MD - Edgewood,4375,Baltimore,2346,10720
31052,us,United States,Nevada,Winchester,US - KY - Winchester,25316,Lexington,2640,8860
26534,us,USA,New Jersey,Orange,US - CA - Orange,797,Anaheim & Buena Park,1481,8059
39770,us,United States of America,Texas,Gresham,US - OR - Gresham,51920,Willamette Valley,513,5959
37330,us,United States of America,Missouri,Hampton,US - GA - Hampton,784,Atlanta,992,3678
35346,us,United States of America,Georgia,Brookhaven,US - MS - Brookhaven,22042,Transylvania,768,3270


## 7. Interpretacion docente

Con el mismo `destination_with_nearest.csv` que usa `hotel-recommender-trainer`, el pipeline de bookings del recomendador llega casi a cobertura total porque construye `reference` con campos DWH normalizados: `hotel_country_dwh`, `hotel_state_dwh`, `hotel_city_dwh`.

Los historicos de este TP se extrajeron desde `analytic.customer_shopping_model` unido a `analytic.hotel_city_location` y traen `city`, `state`, `country`, `country_code`. En esos archivos, varias filas tienen `city` a nivel barrio/distrito/localidad chica y `state` como mercado mas amplio. Por eso no necesariamente van a matchear igual contra un lookup construido con otra definicion de ciudad.

Entonces, para TP2 hay dos caminos correctos:

1. Usar este pretratamiento, reportar coverage por filas, demanda y geografias crudas unicas, y auditar los no-matcheados de mayor demanda.
2. Si se quiere garantizar el mismo coverage que el recomendador, re-extraer los historicos con los mismos campos DWH usados por el pipeline (`hotel_country_dwh`, `hotel_state_dwh`, `hotel_city_dwh`) o regenerar la referencia desde las coordenadas/campos de estos historicos.

Lo que no conviene hacer es subir coverage con fallbacks amplios sin trazabilidad, porque eso puede asignar ciudades a mercados incorrectos.

## 8. Guardar resultado curado (opcional)

Para TP2 pueden guardar una version curada con `destination_final`, `destination_name` y `match_level`. Si lo hacen, conviene que el archivo sea un output derivado y que el notebook/script pueda reproducirlo.

In [8]:
# Ejemplo opcional:
# out_path = ROOT / 'outputs' / 'historicos_con_destino_canonico_sample.csv.gz'
# out_path.parent.mkdir(parents=True, exist_ok=True)
# df_mapped.to_csv(out_path, index=False, compression='gzip')
# out_path
